In [ ]:
import importlib
import sys
import os

def show_backend(label):
    """Force a fresh import and print which backend is loaded."""
    # Wipe any cached ManifoldEM modules so the env-var dispatch re-runs
    for mod in list(sys.modules):
        if mod.startswith("ManifoldEM"):
            del sys.modules[mod]
    from ManifoldEM import DMembeddingII
    env = os.environ.get("MANIFOLDEM_FERGUSON_BACKEND", "<unset>")
    print(f"{label}")
    print(f"  env var: MANIFOLDEM_FERGUSON_BACKEND = {env}")
    print(f"  loaded:  {DMembeddingII.fergusonE.__module__}")
    print()

# 1. Starting state 
os.environ.pop("MANIFOLDEM_FERGUSON_BACKEND", None)
show_backend("1. Default state")

# 2. Switch to UWM 
os.environ["MANIFOLDEM_FERGUSON_BACKEND"] = "uwm"
show_backend("2. After setting env var to 'uwm'")

# 3. Switch back to legacy 
os.environ.pop("MANIFOLDEM_FERGUSON_BACKEND", None)
show_backend("3. After unsetting env var")

In [ ]:
import sys
import os
print("python:", sys.executable)
print("cwd:", os.getcwd())
print("MANIFOLDEM_FERGUSON_BACKEND:", os.environ.get("MANIFOLDEM_FERGUSON_BACKEND", "<unset>"))

In [ ]:
import glob
import os

DIST_DIR = "/mnt/home/aojha/ceph/manifold/ryr1/distances"

# How many files, what format
files = sorted(os.listdir(DIST_DIR))
print(f"Total files in {DIST_DIR}:")
print(f"  count: {len(files)}")
print(f"  first 5: {files[:5]}")
print(f"  last 5: {files[-5:]}")
print()

# Pick the first one and inspect its structure
first_path = os.path.join(DIST_DIR, files[0])
print(f"Inspecting: {first_path}")
print(f"  size: {os.path.getsize(first_path):,} bytes")
print(f"  extension: {os.path.splitext(first_path)[1]}")

In [ ]:
import numpy as np
import h5py

first_path = os.path.join(DIST_DIR, "IMGs_prD_0.h5")

print(f"Inspecting: {first_path}\n")
with h5py.File(first_path, "r") as f:
    print("Keys and shapes:")
    for k in f.keys():
        obj = f[k]
        if hasattr(obj, "shape"):
            print(f"  {k:<15} shape={obj.shape}, dtype={obj.dtype}")
        else:
            print(f"  {k:<15} (group)")
            for kk in obj.keys():
                sub = obj[kk]
                if hasattr(sub, "shape"):
                    print(f"    .{kk:<13} shape={sub.shape}, dtype={sub.dtype}")

**Single-PrD backend comparison (PrD 0 of RyR1).**

Loads one distance matrix, builds the upper-triangle input (including diagonal zeros that act as the kernel-sum floor), and runs both Ferguson backends on the same data: legacy `core.fergusonE` (tanh fit) and UWM `uwm_ferguson_2026` (linear-ramp fit on central 90%).
Prints σ_opt from each method, the ratio between them, and the percent difference. Let us sanity check if the two backends agree before scaling up to all 53 PrDs.

In [ ]:
from ManifoldEM.core import fergusonE as fergusonE_legacy
from ManifoldEM import uwm_ferguson_2026 as uwm
import numpy as np
import h5py

# Load PrD 0
prd_path = os.path.join(DIST_DIR, "IMGs_prD_0.h5")
with h5py.File(prd_path, "r") as f:
    D_matrix = np.asarray(f["D"])

N = D_matrix.shape[0]
print(f"PrD 0: N = {N} images, D shape = {D_matrix.shape}")

# Build input mirroring DMembeddingII production behavior
# Upper triangle including diagonal: the N diagonal zeros provide the
# kernel-sum floor that prevents log(0) at small ε.
iu = np.triu_indices(N, k=0)
yVal = D_matrix[iu]
D_input = np.sqrt(yVal)
n_zeros = int((yVal == 0).sum())
print(f"D_input: {len(D_input)} entries ({n_zeros} zeros + {len(D_input)-n_zeros} nonzero)")
print(f"  nonzero D range: [{D_input[D_input>0].min():.4g}, {D_input.max():.4g}]")

# Production logEps grid
logEps = np.arange(-150.0, 150.2, 0.2)
print(f"logEps grid: {len(logEps)} points from {logEps[0]:.1f} to {logEps[-1]:.1f}")

# LEGACY backend
print("\nLEGACY (core.fergusonE)")
popt_l, logSumWij_l, resnorm_l, R2_l = fergusonE_legacy(D_input, logEps)
log_eps_opt_l = -popt_l[1] / popt_l[0]
sigma_opt_l = float(np.sqrt(2 * np.exp(log_eps_opt_l)))
print(f"  popt           = {popt_l}")
print(f"  log(ε)_opt     = {log_eps_opt_l:.4f}")
print(f"  σ_opt          = {sigma_opt_l:.6g}")
print(f"  R²             = {R2_l:.5f}")

# --- UWM backend, direct call ---
print("\n UWM (uwm_ferguson_2026, direct call)")
Dsq = D_input * D_input
# Match kernel conventions: legacy uses exp(-D²/(2ε)), UWM uses exp(-D²/σ²).
# Equating: σ² = 2ε  →  σ = √2 · exp(logEps/2)
sigma_grid = np.sqrt(2.0) * np.exp(0.5 * logEps)
A_u = np.empty(len(sigma_grid))
for k, s in enumerate(sigma_grid):
    A_u[k] = uwm.A_ij(Dsq, s)[0]
A_u = np.clip(A_u, 1e-300, None)
log_A_u = np.log(A_u)

x_u = np.log(sigma_grid)
tol = 0.05 * np.log(N)
p_central = 90.0
xl, yl_fit, x_mid, y_mid, slope = uwm.fit_ramp(x_u, log_A_u, tol, p_central)

x_mid_s = float(np.atleast_1d(x_mid).ravel()[0])
y_mid_s = float(np.atleast_1d(y_mid).ravel()[0])
slope_s = float(np.atleast_1d(slope).ravel()[0])
sigma_opt_u = float(np.exp(x_mid_s))
log_eps_opt_u = 2.0 * x_mid_s - np.log(2.0)
print(f"  x_mid (log σ)  = {x_mid_s:.4f}")
print(f"  log(ε)_opt     = {log_eps_opt_u:.4f}")
print(f"  σ_opt          = {sigma_opt_u:.6g}")
print(f"  slope (dim)    = {slope_s:.3f}")

# Headline
print("\n Comparison")
ratio = sigma_opt_l / sigma_opt_u
print(f"  σ_legacy        = {sigma_opt_l:.4f}")
print(f"  σ_uwm           = {sigma_opt_u:.4f}")
print(f"  ratio           = {ratio:.4f}")
print(f"  % difference    = {100*(ratio-1):+.2f} %")

**Plot the two fits on top of the empirical curve.**
Empirical Ferguson curve (black dots) overlaid with the legacy tanh fit (blue) and the UWM linear fit (red, solid on the central 90% it actually fit, dotted where extrapolated). Vertical dashed lines mark each method's chosen log(ε)* and the box shows the headline numbers.
Trims the x-axis to ±6 around the inflection (the full logEps grid is mostly flat plateau), and converts UWM's linear fit from (log σ) into (log ε) so both methods share the same x-axis.

In [ ]:
import matplotlib.pyplot as plt

# Trim x-axis to the interesting region (avoid the huge flat plateaus)
view_lo = log_eps_opt_l - 6.0
view_hi = log_eps_opt_l + 6.0
view_mask = (logEps >= view_lo) & (logEps <= view_hi)
# Legacy tanh fit evaluated on the visible logEps range
fit_logEps = logEps[view_mask]
fit_y_legacy = popt_l[3] + popt_l[2] * np.tanh(popt_l[0] * fit_logEps + popt_l[1])
# UWM linear fit: line in (log σ, log A) space → (logEps, log A) space.
# log σ = (log 2 + logEps) / 2, so log A = slope·(log σ) + intercept becomes:
# log A = slope/2 · logEps  +  (slope · log√2 + intercept)
# Recover intercept from the (x_mid, y_mid) point: intercept = y_mid - slope·x_mid
intercept_uwm = y_mid_s - slope_s * x_mid_s
fit_y_uwm = slope_s * 0.5 * (np.log(2.0) + fit_logEps) + intercept_uwm

# UWM's actual linear segment endpoints (the central 90 % of the ramp it fit)
xl_arr = np.atleast_1d(xl).ravel()
xl_logEps = 2.0 * xl_arr - np.log(2.0)
seg_lo, seg_hi = float(xl_logEps.min()), float(xl_logEps.max())

# Plot
fig, ax = plt.subplots(figsize=(11, 6.5))

# Empirical scatter (legacy logSumWij; UWM's log_A_u is nearly identical here so one curve suffices)
ax.scatter(logEps[view_mask], logSumWij_l[view_mask],
           s=20, color="black", alpha=0.55, label="empirical  log Σ exp(-D²/2ε)", zorder=2)

# Legacy tanh fit
ax.plot(fit_logEps, fit_y_legacy,
        color="#1f77b4", lw=2.2, label=f"legacy tanh fit  (R²={R2_l:.4f})", zorder=3)

# UWM linear fit (only show within the segment it was fit to)
seg_mask = (fit_logEps >= seg_lo) & (fit_logEps <= seg_hi)
ax.plot(fit_logEps[seg_mask], fit_y_uwm[seg_mask],
        color="#d62728", lw=2.2, ls="-",
        label=f"UWM linear-on-central-90% ramp", zorder=3)
# Extend dashed beyond the fitted segment so reader sees it's a line
ax.plot(fit_logEps[~seg_mask], fit_y_uwm[~seg_mask],
        color="#d62728", lw=1.0, ls=":", alpha=0.5, zorder=3)

# Vertical lines at each method's chosen log(ε)*
ax.axvline(log_eps_opt_l, color="#1f77b4", ls="--", lw=1.4, alpha=0.8,
           label=f"legacy  log(ε)* = {log_eps_opt_l:.3f}   →   σ = {sigma_opt_l:,.0f}")
ax.axvline(log_eps_opt_u, color="#d62728", ls="--", lw=1.4, alpha=0.8,
           label=f"UWM     log(ε)* = {log_eps_opt_u:.3f}   →   σ = {sigma_opt_u:,.0f}")

# Annotation in upper-left: comparison summary
txt = (
    f"Ratio  σ_legacy / σ_uwm  =  {ratio:.4f}\n"
    f"Δ                       =  {100*(ratio-1):+.2f} %\n"
    f"UWM slope (≈ dim)       =  {slope_s:.2f}"
)
ax.text(0.025, 0.97, txt, transform=ax.transAxes,
        fontsize=10.5, family="monospace", verticalalignment="top",
        bbox=dict(boxstyle="round,pad=0.5", facecolor="white", edgecolor="0.7", alpha=0.95))

ax.set_xlabel(r"$\log(\varepsilon)$", fontsize=13)
ax.set_ylabel(r"$\log \sum_{i,j}\, \exp(-D^2_{ij}\,/\,2\varepsilon)$", fontsize=13)
ax.set_title(f"Ferguson backend comparison for RyR1, PrD 0  (N = {N} images)", fontsize=13.5)
ax.legend(loc="lower right", fontsize=10, framealpha=0.95)
ax.grid(True, alpha=0.3)
ax.set_xlim(view_lo, view_hi)

plt.tight_layout()
plt.savefig("ferguson_compare_RyR1_prD0.png", dpi=200, bbox_inches="tight")
plt.show()
print("\nSaved: ferguson_compare_RyR1_prD0.png")

## Sigma estimation across PrDs: Ferguson legacy vs UWM ramp fit

For every PrD file in `DIST_DIR`, fit the kernel bandwidth `sigma` (equivalently `log_eps`) using two independent methods and tabulate the results side-by-side.

**Input:** `IMGs_prD_*.h5` files, each containing an N×N pairwise squared-distance matrix under key `D`. Files are processed in natural numeric order (prD_2 before prD_10).

**Methods compared:**
- **Ferguson E (legacy):** sigmoid fit to log(kernel sum) vs log(epsilon); `sigma` recovered at the inflection point. Reports R².
- **UWM:** piecewise-linear ramp fit in (log sigma, log A) space; `sigma` taken at the ramp midpoint, ramp slope gives the intrinsic dimension.

**Output:** DataFrame `df` with one row per PrD, columns: `prd`, `N`, `sigma_legacy`, `sigma_uwm`, `log_eps_legacy`, `log_eps_uwm`, `R2_legacy`, `dim_uwm`, `ratio` (legacy/UWM), `pct_diff`, and success flags `legacy_ok` / `uwm_ok`. Failed fits return NaN rather than killing the loop. Progress prints every 10th PrD.

In [ ]:
import pandas as pd
import warnings
import time

# Sort PrD files numerically (IMGs_prD_2 before IMGs_prD_10) since alphabetical sort would break ordering
def prd_index(fname):
    return int(fname.replace("IMGs_prD_", "").replace(".h5", ""))

files = sorted(os.listdir(DIST_DIR), key=prd_index)
print(f"Processing {len(files)} PrD files...")

records = []
t0 = time.time()

for fname in files:
    prd = prd_index(fname)
    path = os.path.join(DIST_DIR, fname)
    # Load pairwise distance matrix D (N x N, squared distances)
    with h5py.File(path, "r") as f:
        D_matrix = np.asarray(f["D"])
    N = D_matrix.shape[0]
    # Flatten upper triangle including diagonal - input to both fitting methods
    iu = np.triu_indices(N, k=0)
    D_input = np.sqrt(D_matrix[iu])   # distances
    Dsq = D_input * D_input            # squared distances (needed by UWM)
    # Method 1: Ferguson E-statistic (legacy) 
    # Fits log(sum_kernel) vs log(epsilon) to a sigmoid-like curve;
    # sigma is read off from the inflection point. Returns popt, R2.
    try:
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            popt_l, _, _, R2_l = fergusonE_legacy(D_input, logEps)
        # Recover sigma and log(epsilon) at the inflection from popt = [slope, intercept]
        sigma_l = float(np.sqrt(2 * np.exp(-popt_l[1] / popt_l[0])))
        log_eps_l = -popt_l[1] / popt_l[0]
        leg_ok = True
    except Exception as e:
        sigma_l, log_eps_l, R2_l = np.nan, np.nan, np.nan
        leg_ok = False
        leg_err = str(e)[:60]
    # Method 2: UWM (uniform-weight matrix ramp fit) 
    # Builds A(sigma) curve, fits a piecewise ramp in log-log space;
    # sigma is the midpoint of the linear ramp, slope = intrinsic dimension.
    try:
        # sigma grid mirrors logEps via sigma = sqrt(2) * exp(logEps/2)
        sigma_grid = np.sqrt(2.0) * np.exp(0.5 * logEps)
        # Compute A(sigma) for every grid point; clip to avoid log(0)
        A_u = np.array([uwm.A_ij(Dsq, s)[0] for s in sigma_grid])
        A_u = np.clip(A_u, 1e-300, None)
        # Fit piecewise-linear ramp; tol scales with log(N)
        x_u, y_u = np.log(sigma_grid), np.log(A_u)
        tol = 0.05 * np.log(N)
        xl, yl_fit, x_mid, y_mid, slope = uwm.fit_ramp(x_u, y_u, tol, 90.0)
        # Extract scalars (fit_ramp may return arrays)
        x_mid_s = float(np.atleast_1d(x_mid).ravel()[0])
        slope_s = float(np.atleast_1d(slope).ravel()[0])
        sigma_u = float(np.exp(x_mid_s))           # sigma at ramp midpoint
        log_eps_u = 2.0 * x_mid_s - np.log(2.0)    # convert back to logEps scale
        uwm_ok = True
    except Exception as e:
        sigma_u, log_eps_u, slope_s = np.nan, np.nan, np.nan
        uwm_ok = False
        uwm_err = str(e)[:60]
    # Store per-PrD results — ratio and pct_diff quantify legacy vs UWM agreement
    records.append({
        "prd":            prd,
        "N":              N,
        "sigma_legacy":   sigma_l,
        "sigma_uwm":      sigma_u,
        "log_eps_legacy": log_eps_l,
        "log_eps_uwm":    log_eps_u,
        "R2_legacy":      R2_l,
        "dim_uwm":        slope_s,                     # intrinsic dim from ramp slope
        "ratio":          sigma_l / sigma_u if (leg_ok and uwm_ok) else np.nan,
        "pct_diff":       100*(sigma_l/sigma_u - 1) if (leg_ok and uwm_ok) else np.nan,
        "legacy_ok":      leg_ok,
        "uwm_ok":         uwm_ok,
    })

    # Sparse progress print: every 10th PrD plus the final one
    if prd % 10 == 0 or prd == len(files) - 1:
        print(f"  PrD {prd:3d}  N={N:4d}  σ_leg={sigma_l:10.1f}  σ_uwm={sigma_u:10.1f}  "
              f"ratio={sigma_l/sigma_u:.4f}  Δ={100*(sigma_l/sigma_u-1):+6.2f}%  dim={slope_s:.2f}")
elapsed = time.time() - t0
# Assemble final DataFrame sorted by PrD index
df = pd.DataFrame(records).sort_values("prd").reset_index(drop=True)
print(f"\nDone in {elapsed:.1f}s.  {df['legacy_ok'].sum()}/{len(df)} legacy ok, "
      f"{df['uwm_ok'].sum()}/{len(df)} UWM ok.")
df

In [ ]:
# Diagnose the failures
for prd in [50, 51, 52]:
    path = os.path.join(DIST_DIR, f"IMGs_prD_{prd}.h5")
    with h5py.File(path, "r") as f:
        D = np.asarray(f["D"])
    n_neg = (D < 0).sum()
    n_total = D.size
    print(f"PrD {prd}: N={D.shape[0]}, {n_neg} negative entries ({100*n_neg/n_total:.3f}%), "
          f"min={D.min():.4g}, max={D.max():.4g}")

## Visualize legacy vs UWM agreement across all PrDs

Two-panel diagnostic figure summarizing the per-PrD comparison from the previous cell.

**Panel 1 - Scatter `sigma_legacy` vs `sigma_uwm`:**
Each point is one PrD, colored by `N` (number of images). A y=x diagonal marks perfect agreement; dotted ±1% and ±3% reference lines bracket acceptable bias. Points should hug the diagonal if the two methods are consistent.

**Panel 2 - Percent difference vs N:**
`pct_diff = 100*(sigma_legacy/sigma_uwm - 1)` plotted against PrD size. Green and yellow bands mark the ±1% and ±3% tolerance regions. Reveals whether disagreement scales with sample size (small-N noise vs systematic bias). An inset text box reports the PrD count, fit failures, median/mean/max percent difference, and the range of UWM intrinsic-dimension estimates.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Use only the successful PrDs
df_ok = df.dropna(subset=["sigma_legacy", "sigma_uwm"]).copy()
# Figure with two panels side by side
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6.2))
# Panel 1: σ_legacy vs σ_uwm scatter, colored by N
sc = ax1.scatter(df_ok["sigma_legacy"], df_ok["sigma_uwm"],
                 c=df_ok["N"], cmap="viridis", s=70, edgecolors="black",
                 linewidth=0.6, zorder=3)
# y=x diagonal
lo = min(df_ok["sigma_legacy"].min(), df_ok["sigma_uwm"].min()) * 0.995
hi = max(df_ok["sigma_legacy"].max(), df_ok["sigma_uwm"].max()) * 1.005
ax1.plot([lo, hi], [lo, hi], "k--", lw=1.2, alpha=0.5, label="y = x  (perfect agreement)")
# ±1%, ±3% reference lines
xx = np.linspace(lo, hi, 100)
ax1.plot(xx, 1.01*xx, color="0.55", ls=":", lw=1.0, alpha=0.7)
ax1.plot(xx, 0.99*xx, color="0.55", ls=":", lw=1.0, alpha=0.7, label="±1%")
ax1.plot(xx, 1.03*xx, color="0.75", ls=":", lw=1.0, alpha=0.7)
ax1.plot(xx, 0.97*xx, color="0.75", ls=":", lw=1.0, alpha=0.7, label="±3%")
cb = plt.colorbar(sc, ax=ax1, label="N (images per PrD)")
ax1.set_xlabel(r"$\sigma$ — legacy (tanh fit)", fontsize=12)
ax1.set_ylabel(r"$\sigma$ — UWM 2026 (linear ramp)", fontsize=12)
ax1.set_title(f"σ agreement across {len(df_ok)} RyR1 PrDs", fontsize=13)
ax1.legend(loc="lower right", fontsize=9)
ax1.grid(True, alpha=0.3)
ax1.set_xlim(lo, hi)
ax1.set_ylim(lo, hi)
# Panel 2: % difference vs N 
ax2.scatter(df_ok["N"], df_ok["pct_diff"], c=df_ok["N"], cmap="viridis",
            s=70, edgecolors="black", linewidth=0.6, zorder=3)
ax2.axhline(0, color="k", lw=0.8, alpha=0.5)
ax2.axhspan(-1, 1, color="green", alpha=0.1, label="±1% band")
ax2.axhspan(-3, 3, color="yellow", alpha=0.08, label="±3% band")
ax2.set_xlabel("N (images per PrD)", fontsize=12)
ax2.set_ylabel(r"$100 \cdot (\sigma_{\rm legacy} / \sigma_{\rm UWM} - 1)$  [%]", fontsize=12)
ax2.set_title("Disagreement scales with PrD size", fontsize=13)
ax2.legend(loc="lower left", fontsize=9)
ax2.grid(True, alpha=0.3)
# Annotation with summary stats
median_pct = df_ok["pct_diff"].median()
max_abs_pct = df_ok["pct_diff"].abs().max()
mean_pct = df_ok["pct_diff"].mean()
txt = (
    f"PrDs compared:   {len(df_ok)} / {len(df)}\n"
    f"failed:          {len(df) - len(df_ok)} (negative D entries)\n"
    f"median Δ:        {median_pct:+.2f} %\n"
    f"mean Δ:          {mean_pct:+.2f} %\n"
    f"max |Δ|:         {max_abs_pct:.2f} %\n"
    f"UWM slope range: {df_ok['dim_uwm'].min():.2f} – {df_ok['dim_uwm'].max():.2f}"
)
ax2.text(0.025, 0.97, txt, transform=ax2.transAxes,
         fontsize=10, family="monospace", verticalalignment="top",
         bbox=dict(boxstyle="round,pad=0.5", facecolor="white", edgecolor="0.7", alpha=0.95))
fig.suptitle("Ferguson backend cross-validation for RyR1 (53 PrDs)",
             fontsize=14, fontweight="bold", y=1.00)
plt.tight_layout()
plt.savefig("ferguson_master_RyR1.png", dpi=400, bbox_inches="tight")
plt.show()
print("\nSaved: ferguson_master_RyR1.png")
# Also save the per-PrD CSV
df.to_csv("ferguson_compare_RyR1_per_prd.csv", index=False)
print("Saved: ferguson_compare_RyR1_per_prd.csv")